# Diffusion 기반 의료 AI 방어 기법 - 학습 & 공격 실행
**Google Colab GPU 환경에서 실행**

순서:
1. GitHub 레포 클론
2. CheXpert 데이터 다운로드 (Kaggle)
3. Week 1: 베이스라인 분류기 학습
4. Week 2: FGSM/PGD 공격 + FFT 분석

## 0. 환경 설정

In [ ]:
# GPU 확인
!nvidia-smi

In [ ]:
# GitHub 레포 클론
!git clone https://github.com/jiwoo1105/AISecurity.git
%cd AISecurity

In [ ]:
# 패키지 설치
!pip install -q lpips pyyaml

## 1. CheXpert 데이터 다운로드 (Kaggle)

In [ ]:
# Kaggle API 키 설정
# Kaggle > Settings > API > Create New Token 에서 kaggle.json 다운로드 후 업로드
from google.colab import files
uploaded = files.upload()  # kaggle.json 업로드

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# CheXpert-v1.0-small 다운로드 (~11GB, 약 5분)
!pip install -q kaggle
!kaggle datasets download -d ashery/chexpert -p data/chexpert
!cd data/chexpert && unzip -q chexpert.zip && rm chexpert.zip

# 폴더 구조 확인
!ls data/chexpert/

In [ ]:
# 데이터 경로 맞추기
import os
import pandas as pd

df = pd.read_csv('data/chexpert/train.csv')
print('CSV 경로 예시:', df['Path'].iloc[0])

if 'CheXpert-v1.0-small' in df['Path'].iloc[0]:
    os.makedirs('data/chexpert/CheXpert-v1.0-small', exist_ok=True)
    if not os.path.exists('data/chexpert/CheXpert-v1.0-small/train'):
        os.symlink(os.path.abspath('data/chexpert/train'), 'data/chexpert/CheXpert-v1.0-small/train')
    if not os.path.exists('data/chexpert/CheXpert-v1.0-small/valid'):
        os.symlink(os.path.abspath('data/chexpert/valid'), 'data/chexpert/CheXpert-v1.0-small/valid')
    print('심볼릭 링크 생성 완료')

## 2. 데이터 확인

In [ ]:
df = pd.read_csv('data/chexpert/train.csv')
frontal = df[df['Frontal/Lateral'] == 'Frontal']

print(f'전체 이미지: {len(df):,}장')
print(f'Frontal만: {len(frontal):,}장')
print()
print('Pneumonia 라벨 분포 (Frontal):')
print(frontal['Pneumonia'].value_counts(dropna=False))

In [ ]:
# 샘플 이미지 확인
import matplotlib.pyplot as plt
from PIL import Image

has_label = frontal[frontal['Pneumonia'].notna() | (frontal['No Finding'] == 1.0)]
sample_paths = has_label.head(4)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i, (_, row) in enumerate(sample_paths.iterrows()):
    img = Image.open(os.path.join('data/chexpert', row['Path']))
    label = 'Pneumonia' if row['Pneumonia'] in [1.0, -1.0] else 'Normal'
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(label, fontsize=14)
    axes[i].axis('off')

plt.suptitle('CheXpert X-ray Samples', fontsize=16)
plt.tight_layout()
plt.show()

## 3. Week 1: 베이스라인 분류기 학습

In [ ]:
import sys
sys.path.insert(0, '.')

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.notebook import tqdm

from models.classifier import PneumoniaClassifier
from data.dataset import get_dataloaders
from evaluation.metrics import compute_accuracy, compute_auc, compute_ece

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# 데이터 로드
train_loader, val_loader, test_loader = get_dataloaders(
    csv_path='data/chexpert/train.csv',
    data_dir='data/chexpert',
    image_size=224,
    batch_size=32,
    num_workers=2,
    seed=42,
)

In [ ]:
# 모델 + 학습 설정
model = PneumoniaClassifier(name='resnet50', pretrained=True).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = CosineAnnealingLR(optimizer, T_max=10)

EPOCHS = 10
best_auc = 0.0
history = {'train_loss': [], 'val_acc': [], 'val_auc': []}

In [ ]:
# 학습 루프
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    scheduler.step()
    avg_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            probs = torch.sigmoid(model(images))
            all_probs.append(probs.cpu())
            all_labels.append(labels.cpu())

    all_probs = torch.cat(all_probs)
    all_labels = torch.cat(all_labels)
    preds = (all_probs >= 0.5).long()

    val_acc = compute_accuracy(preds, all_labels)
    val_auc = compute_auc(all_probs, all_labels)

    history['train_loss'].append(avg_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)

    print(f'Epoch {epoch}: Loss={avg_loss:.4f} | Val Acc={val_acc:.4f} | Val AUC={val_auc:.4f}')

    if val_auc > best_auc:
        best_auc = val_auc
        os.makedirs('models/checkpoints', exist_ok=True)
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_auc': val_auc,
            'val_acc': val_acc,
        }, 'models/checkpoints/resnet50_best.pth')
        print(f'  Best model saved! AUC: {val_auc:.4f}')

print(f'\nBest Val AUC: {best_auc:.4f}')

In [ ]:
# 학습 곡선 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'], 'b-')
axes[0].set_title('Train Loss')
axes[0].set_xlabel('Epoch')

axes[1].plot(history['val_acc'], 'g-')
axes[1].set_title('Val Accuracy')
axes[1].set_xlabel('Epoch')

axes[2].plot(history['val_auc'], 'r-')
axes[2].set_title('Val AUC')
axes[2].set_xlabel('Epoch')
axes[2].axhline(y=0.85, color='gray', linestyle='--', label='Target 0.85')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Test 평가
checkpoint = torch.load('models/checkpoints/resnet50_best.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Testing'):
        images, labels = images.to(device), labels.to(device)
        probs = torch.sigmoid(model(images))
        all_probs.append(probs.cpu())
        all_labels.append(labels.cpu())

all_probs = torch.cat(all_probs)
all_labels = torch.cat(all_labels)
preds = (all_probs >= 0.5).long()

test_acc = compute_accuracy(preds, all_labels)
test_auc = compute_auc(all_probs, all_labels)
test_ece = compute_ece(all_probs, all_labels)

print(f'\n{"="*40}')
print(f'Test Results (Baseline ResNet-50)')
print(f'{"="*40}')
print(f'Clean Accuracy: {test_acc:.4f}')
print(f'AUC:            {test_auc:.4f}')
print(f'ECE:            {test_ece:.4f}')

## 4. Week 2: FGSM/PGD 공격 실행

In [ ]:
from attacks.fgsm import FGSM
from attacks.pgd import PGD
from evaluation.metrics import compute_asr

epsilons = [0.01, 0.02, 0.04, 0.08]
results = []

In [ ]:
# FGSM 공격
print('=== FGSM Attack ===')
for eps in epsilons:
    attacker = FGSM(model, epsilon=eps)
    result = attacker.attack_batch(test_loader, device)
    asr = compute_asr(result['orig_preds'], result['adv_preds'])
    results.append({'attack': 'FGSM', 'epsilon': eps, 'asr': asr, 'result': result})
    print(f'  eps={eps:.2f}  ASR: {asr:.2%}')

In [ ]:
# PGD 공격
print('=== PGD Attack (20 steps) ===')
for eps in epsilons:
    attacker = PGD(model, epsilon=eps, step_size=0.01, num_steps=20)
    result = attacker.attack_batch(test_loader, device)
    asr = compute_asr(result['orig_preds'], result['adv_preds'])
    results.append({'attack': 'PGD', 'epsilon': eps, 'asr': asr, 'result': result})
    print(f'  eps={eps:.2f}  ASR: {asr:.2%}')

In [ ]:
# 공격 결과 표
print(f'\n{"="*50}')
print(f'{"Attack":<8} {"Epsilon":<10} {"ASR":<10}')
print(f'{"="*50}')
for r in results:
    print(f'{r["attack"]:<8} {r["epsilon"]:<10.2f} {r["asr"]:<10.2%}')

In [ ]:
# ASR 비교 그래프
fig, ax = plt.subplots(figsize=(8, 5))

fgsm_asr = [r['asr'] for r in results if r['attack'] == 'FGSM']
pgd_asr = [r['asr'] for r in results if r['attack'] == 'PGD']

ax.plot(epsilons, fgsm_asr, 'bo-', label='FGSM', linewidth=2, markersize=8)
ax.plot(epsilons, pgd_asr, 'rs-', label='PGD (20 steps)', linewidth=2, markersize=8)
ax.set_xlabel('Epsilon', fontsize=12)
ax.set_ylabel('Attack Success Rate (ASR)', fontsize=12)
ax.set_title('FGSM vs PGD: Attack Success Rate', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
# 원본 vs 공격 이미지 비교
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

fgsm_r = [r for r in results if r['attack'] == 'FGSM' and r['epsilon'] == 0.04][0]['result']
pgd_r = [r for r in results if r['attack'] == 'PGD' and r['epsilon'] == 0.04][0]['result']

for i in range(4):
    axes[0, i].imshow(fgsm_r['originals'][i][0].numpy(), cmap='gray')
    axes[0, i].set_title(f'Original (pred={fgsm_r["orig_preds"][i].item()})')
    axes[0, i].axis('off')

    axes[1, i].imshow(fgsm_r['adversarials'][i][0].numpy(), cmap='gray')
    axes[1, i].set_title(f'FGSM (pred={fgsm_r["adv_preds"][i].item()})')
    axes[1, i].axis('off')

    axes[2, i].imshow(pgd_r['adversarials'][i][0].numpy(), cmap='gray')
    axes[2, i].set_title(f'PGD (pred={pgd_r["adv_preds"][i].item()})')
    axes[2, i].axis('off')

plt.suptitle('Original vs Adversarial Images (eps=0.04)', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# FFT 스펙트럼 비교
import torch.fft as fft_module

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

orig = fgsm_r['originals'][0][0]
fgsm_adv = fgsm_r['adversarials'][0][0]
pgd_adv = pgd_r['adversarials'][0][0]

for i, (img, title) in enumerate([(orig, 'Original'), (fgsm_adv, 'FGSM'), (pgd_adv, 'PGD')]):
    axes[0, i].imshow(img.numpy(), cmap='gray')
    axes[0, i].set_title(title, fontsize=14)
    axes[0, i].axis('off')

    f = fft_module.fft2(img)
    magnitude = torch.log1p(torch.abs(fft_module.fftshift(f)))
    axes[1, i].imshow(magnitude.numpy(), cmap='hot')
    axes[1, i].set_title(f'{title} - FFT', fontsize=14)
    axes[1, i].axis('off')

plt.suptitle('FFT Frequency Spectrum Comparison', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# 주파수 대역별 에너지 비교
from defense.fft_analyzer import FFTAnalyzer

analyzer = FFTAnalyzer()

orig_bands = analyzer.compute_frequency_bands(orig.unsqueeze(0).unsqueeze(0))
fgsm_bands = analyzer.compute_frequency_bands(fgsm_adv.unsqueeze(0).unsqueeze(0))
pgd_bands = analyzer.compute_frequency_bands(pgd_adv.unsqueeze(0).unsqueeze(0))

band_names = ['Low', 'Mid', 'High']
orig_e = [orig_bands[k].mean().item() for k in ['low', 'mid', 'high']]
fgsm_e = [fgsm_bands[k].mean().item() for k in ['low', 'mid', 'high']]
pgd_e = [pgd_bands[k].mean().item() for k in ['low', 'mid', 'high']]

x = range(3)
w = 0.25
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar([i - w for i in x], orig_e, w, label='Original', color='#2196F3')
ax.bar(x, fgsm_e, w, label='FGSM', color='#FF9800')
ax.bar([i + w for i in x], pgd_e, w, label='PGD', color='#F44336')
ax.set_xticks(x)
ax.set_xticklabels(band_names)
ax.set_ylabel('Mean Energy')
ax.set_title('Frequency Band Energy: Original vs Attacks')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print('\n-> High 영역에서 FGSM/PGD 에너지 증가 = 고주파 노이즈 공격 특징')
print('-> Week 3 DiffAttack은 Mid 영역이 증가할 것으로 예상')

## 5. 결과 저장 & 다운로드

In [ ]:
import csv

os.makedirs('results', exist_ok=True)

with open('results/attack_results.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['attack', 'epsilon', 'asr'])
    for r in results:
        writer.writerow([r['attack'], r['epsilon'], f"{r['asr']:.4f}"])

with open('results/baseline_eval.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['model', 'clean_accuracy', 'auc', 'ece'])
    writer.writerow(['resnet50', f'{test_acc:.4f}', f'{test_auc:.4f}', f'{test_ece:.4f}'])

print('Results saved!')

In [ ]:
# 학습된 모델 + 결과 다운로드
from google.colab import files
files.download('models/checkpoints/resnet50_best.pth')
files.download('results/baseline_eval.csv')
files.download('results/attack_results.csv')